# പാഠം 03 - ഏജന്റിക് ഡിസൈൻ മാതൃകകൾ

ഈ പാഠത്തിൽ, ഫലപ്രദമായ AI ഏജന്റുകൾ നിർമ്മിക്കാൻ മൂന്ന് അടിസ്ഥാന ഡിസൈൻ മാതൃകകളെക്കുറിച്ച് പഠിക്കും:

1. **സ്വച്ഛമായ ഏജന്റ് നിർദ്ദേശങ്ങൾ** — ഏജന്റിന്റെ പെരുമാറ്റം നയിക്കുന്ന, സുഷ്ടമായ, പദവി നിർവചിക്കുന്ന പ്രൊമ്പ്റ്റുകൾ രൂപകല്‍പ്പന ചെയ്യൽ  
2. **പിഡാന്റിക് മോഡലുകളുള്ള ഘടിത ഔട്ട്പുട്ട്** — ഏജന്റുകൾ കരുതലോടും സ്ഥിരീകരിക്കപ്പെട്ടതുമായ ഡാറ്റ തിരികെ നൽകുന്നത് ഉറപ്പാക്കൽ  
3. **ഏകകാലിക ഉത്തരവാദിത്വ ഏജന്റുകൾ** — ഓരോ ഏജന്റും ഒരു കാര്യത്തിൽ പ്രാവിണ്യം കാണിക്കുന്ന വിധം രൂപകല്‌പന ചെയ്യൽ

നാം ഓരോ മാതൃകയും **യാത്രാസ്ഥല ശുപാർശദാതാവ്** ഘട്ടം അടിസ്ഥാനമാക്കിയുള്ള സാഹചര്യത്തിൽ പ്രയോഗിച്ച്, എത്തിപ്പെടൽസ്ഥലങ്ങൾ ശുപാർശ ചെയ്യുകയും, ലഭ്യത പരിശോധിക്കുകയും, ലജിസ്റ്റിക്സ് കൈകാര്യം ചെയ്യുകയും ചെയ്യുന്ന ഒരു സിസ്റ്റം ക്രമീകരിക്കുകയാകും.


## ക്രമീകരണം


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity pydantic --quiet

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated
from pydantic import BaseModel
from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

## Pattern 1: വ്യക്തമായ ഏജന്റ് നിർദ്ദേശങ്ങൾ

അതിathyadha_prabhavakaramaaya പാറ്റേൺ ഏറ്റവും ലളിതവുമാണ്: നിങ്ങളുടെ ഏജന്റിനായി വ്യക്തവും വിശദവുമായ നിർദേശങ്ങൾ എഴുതുക.

നല്ല നിർദേശങ്ങൾ നിർവ്വചിക്കുന്നു:
- **ആരാണ്** ഏജന്റ് (പേഴ്സണയും ടോൺ)
- **ഏതെന്ത്** ചെയ്യണം (കടന്നുപോകുന്ന ഉത്തരവാദിത്വങ്ങൾ)
- **എങ്ങനെ** പെരുമാറണം (നിയമങ്ങൾക്കുള്ള പൂട്ടുകളും ശൈലിയും)

താഴെ, നാം ഒരു യാത്രാ കോൺസിയർജ് ഏജന്റിനെ സൃഷ്ടിക്കുന്നു, വ്യക്തമായ നിർദേശങ്ങളോടെ, അത് ഓരോ പ്രതികരണത്തെയും രൂപപ്പെടുത്തുന്നു.


In [ ]:
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

agent = await provider.create_agent(
    name="TravelConcierge",
    instructions="""You are a luxury travel concierge named Alex. Your role is to:
1. Understand the traveler's preferences (budget, climate, activities)
2. Check destination availability before making recommendations
3. Provide detailed, personalized travel suggestions
4. Always mention visa requirements and best travel seasons
Be warm, professional, and enthusiastic about travel.""",
)

response = await agent.run(
    "I'd love a week-long vacation somewhere with great food and history. Budget around $2500."
)
print(response)

## Pattern 2: Pydantic മോഡലുകളുമായി ഘടനാപരമായ ഔട്ട്പുട്ട്

സൗജന്യരീതിയായി എഴുതുന്ന വാചകങ്ങൾ സംഭാഷണത്തിനുള്ളതിനുള്ളതാണ്, бірақ താഴോട്ടുള്ള സിസ്റ്റങ്ങൾക്ക് ഘടിതമായ ഡാറ്റ ആവശ്യമുണ്ട്.
**Pydantic മോഡലുകൾ** ഒരു **ടൂൾ ഫംഗ്ഷനുമായ** കൂട്ടിച്ചേർത്ത്, നാം:

- ഏജന്റിന്റെ ഔട്ട്പുട്ടിനുള്ള കൃത്യമായ സ്കീമ നിർവ്വചിക്കാം
- പ്രതികരണങ്ങളെ സ്വയം സാധുതപ്പെടുത്താം
- ഏജന്റ് ഫലങ്ങളെ ആപ്ലിക്കേഷൻ ലജിക്കിൽ വിശ്വസനീയമായി സംയോജിപ്പിക്കാം

നാം ഒരു ടൂൾ കൂടി പരിചയപ്പെടുന്നു, അത് ലക്ഷ്യസ്ഥലത്തിന്റെ വിവരങ്ങൾ തിരികെ നൽകുന്നു, അതിലൂടെ ഏജന്റ് തന്റെ ശിപാർശകൾ യാഥാർത്ഥ്യത്തിലുള്ള ഡാറ്റയിൽ ആധാരപ്പെടുത്തുന്നു.


In [ ]:
class DestinationRecommendation(BaseModel):
    destination: str
    available: bool
    best_season: str
    highlights: list[str]
    estimated_budget_usd: int


class TravelRecommendations(BaseModel):
    recommendations: list[DestinationRecommendation]
    personalized_note: str


@tool(approval_mode="never_require")
def get_destination_details(destination: Annotated[str, "The destination to look up"]) -> str:
    """Get details about a vacation destination."""
    details = {
        "Barcelona": "Available. Best: May-Jun. Beach, architecture, nightlife. ~$2000/week",
        "Tokyo": "Available. Best: Mar-Apr. Culture, food, technology. ~$2500/week",
        "Cape Town": "Not available. Best: Nov-Mar. Nature, wine, adventure. ~$1800/week",
    }
    return details.get(destination, f"{destination}: No information available.")


structured_agent = await provider.create_agent(
    name="StructuredTravelExpert",
    instructions="You are a travel expert. Recommend destinations based on traveler preferences. Use the get_destination_details tool.",
    tools=[get_destination_details],
)

response = await structured_agent.run(
    "Recommend 3 destinations for a culture-loving traveler with a $2500 budget"
)

if response:
    print(response)

## Pattern 3: Single Responsibility Agents

സങ്കീർണ്ണമായ പ്രവൃത്തികളെ ഓരോ ഏജന്റ് ഫോകസ് ചെയ്ത് ഓരോ ഉത്തരവാദിത്വത്തോടെയും വിഭജിക്കുന്നത് ഗുണകരമാണ്:

- സ്ഥലങ്ങളും ലഭ്യതയും അറിയുന്ന **Destination Expert**
- വിമാനയാത്രകൾ, ഹോട്ടലുകൾ, യാത്രാപദ്ധതികൾ എന്നിവ കൈകാര്യം ചെയ്യുന്ന **Logistics Planner**

ഇത് സോഫ്റ്റ്വെയർ എഞ്ചിനീയറിങ്ങിലെ *separation of concerns* എന്ന സിദ്ധാന്തം അനുകരിക്കുന്നു — ഓരോ ഏജന്റും സ്വതന്ത്രമായി പരിശോധന ചെയ്യാനും പരിപാലിക്കാനും മെച്ചപ്പെടുത്താനുമുള്ള സ്ഥിരത കൂടുതലാണ്.


In [ ]:
destination_agent = await provider.create_agent(
    name="DestinationExpert",
    tools=[get_destination_details],
    instructions="""You are a destination research specialist. Your only job is to:
1. Evaluate destinations based on traveler preferences
2. Check availability using the provided tool
3. Return a short ranked list with pros/cons
Do NOT discuss flights, hotels, or logistics — another agent handles that.""",
)

logistics_agent = await provider.create_agent(
    name="LogisticsPlanner",
    instructions="""You are a travel logistics planner. Your only job is to:
1. Create a day-by-day itinerary for the chosen destination
2. Suggest flight and hotel options within the stated budget
3. Note visa requirements and travel insurance recommendations
Do NOT recommend destinations — another agent handles that.""",
)

# Step 1: Destination Expert picks the best options
dest_response = await destination_agent.run(
    "I want a week of culture and food for under $2500. Where should I go?"
)
print("=== Destination Expert ===")
print(dest_response)

# Step 2: Logistics Planner builds the trip plan
logistics_response = await logistics_agent.run(
    f"Plan a week-long trip based on this recommendation:\n{dest_response}"
)
print("\n=== Logistics Planner ===")
print(logistics_response)

## സംഗ്രഹം

ഈ പാഠത്തിൽ നാം യാത്രാ ശിപാർശാകർ സ്കenario-വിൽ മൂന്ന് ഏജന്റിക് ഡിസൈൻ മാതൃകകൾ പ്രയോഗിച്ചു:

| മാതൃക | പ്രധാന ആശയം | ഗുണം |
|---|---|---|
| **സ്പഷ്ടമായ നിർദേശങ്ങൾ** | വ്യക്തിത്വം, ഉത്തരവാദിത്വങ്ങൾ, പരിധികൾ മുമ്പേ നിർദേശം നൽകുക | സുസ്ഥിരമായ, ബ്രാൻഡിന്റെ അനുസൃതമായ ഏജന്റ് പെരുമാറ്റം |
| **ഘടനാപരമായ നിർമാണം** | പ്രതികരണ ഫോർമാറ്റായി Pydantic മോഡലുകൾ ഉപയോഗിക്കുക | അംഗീകൃതവും യന്ത്രം വായിക്കാൻ കഴിയുന്ന ഫലങ്ങൾ |
| **ഒറ്റ ഉത്തരവാദിത്വം** | ഓരോ ഏജന്റിനും ഒരു കേന്ദ്രീകൃത ജോബ് നൽകുക | പരീക്ഷിക്കാനും പരിപാലിക്കാനും സംയോജിപ്പിക്കാനുമുള്ള ലളിതം |

ഈ മാതൃകകൾ സ്വാഭാവികമായി സംയോജിപ്പിക്കാം — സുസ്പഷ്ട നിർദേശങ്ങൾ ഘടനാപരമായ നിർമാണത്തോടെ കൂടി ഒറ്റ ഉത്തരവാദിത്വ ഏജന്റിനകം സംയോജിപ്പിച്ച് ഉറച്ച, നിർമ്മിത-സജ്ജമായ സംവിധാനങ്ങൾ നിർമ്മിക്കാം.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**പരാമർശം**:  
ഈ ഡോക്യുമെന്റ് AI വിവർത്തന സേവനമായ [Co-op Translator](https://github.com/Azure/co-op-translator) ഉപയോഗിച്ച് വിവർത്തനം ചെയ്തതാണ്. നമുക്ക് കൃത്യതയ്ക്ക് ശ്രമിച്ചാലും, സ്വയമേവ ചെയ്ത വിവർത്തനങ്ങളിൽ പിശകുകളോ തെറ്റുകളോ ഉണ്ടാകാമെന്ന് ദയവായി ശ്രദ്ധിക്കുക. ഈ ഡോക്യുമെന്റിന്റെ മൗലിക ഭാഷയിലെ ഓറിയജിനൽ രൂപമാണ് പ്രാമാണികമായ ഉറവിടം എന്ന് കണക്കാക്കേണ്ടതാണ്. നിർണായക വിവരങ്ങൾക്കായി പ്രൊഫഷണൽ മനുഷ്യ വിവർത്തനം ശുപാർശ ചെയ്യപ്പെടുന്നു. ഈ വിവർത്തനം ഉപയോഗിച്ചതിലൂടെ ഉണ്ടായേക്കാവുന്ന തെറ്റു മനസ്സിലാക്കലുകൾക്ക് ഞങ്ങൾ ഉത്തരവാദിത്വം ഏറ്റെടുക്കുന്നില്ല.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
